# 10 — Checkpoint and resume
CrewAI can save Crew execution state after tasks. Start the script, stop it
after the first task finishes, then pass a checkpoint file to --resume.
A checkpoint helps avoid repeating completed *research*; side effects still
need an idempotent action boundary like the mock email from lesson 09.



**Basic lesson 10 of 13.**


## Problem

A long task may stop after completing useful work.


## Concept

CrewAI checkpoints can restore progress; side effects still require idempotency.

A checkpoint remembers completed Crew work after interruption. It does not create exactly-once semantics for side effects, so a stable action ID and an idempotent sink remain necessary.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume   ◀ YOU ARE HERE
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly
```

Every lesson adds one row to this system. The rows below the marker are still the same code from earlier lessons.


## Architecture of this lesson

```text
task 1 ──▶ checkpoint file ──▶ task 2
               │ interruption         ▲
               └──▶ restore path ──────┘
side effect ──▶ separate idempotency key (checkpoint alone is insufficient)
```

task 1 → checkpoint → interruption → restore → task 2. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Create a checkpoint-enabled Crew with two tasks

Create a checkpoint-enabled Crew with two tasks.


In [ ]:
import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

import workshop  # configure project-local CrewAI storage
from crewai import Agent, CheckpointConfig, Crew, Process, Task

from workshop.common import model

parser = argparse.ArgumentParser()
parser.add_argument("--resume", type=Path, help="Path to a CrewAI .checkpoints JSON file")
args, _ = parser.parse_known_args()

researcher = Agent(role="Researcher", goal="Find two facts", backstory="Careful researcher", llm=model())
writer = Agent(role="Writer", goal="Summarize research", backstory="Clear communicator", llm=model())
crew = Crew(
    agents=[researcher, writer],
    tasks=[
        Task(description="List two facts about a cautious AI pilot", expected_output="Two bullets", agent=researcher),
        Task(description="Summarize the two facts", expected_output="A short paragraph", agent=writer),
    ],
    process=Process.sequential,
    checkpoint=True,
)



## Run or resume
The checkpoint path is printed by the workshop instructions after an
interrupted run. Do not invent a checkpoint file; pass the one CrewAI saved.



## Step 2 — Run normally or restore only from a real saved checkpoint path

Run normally or restore only from a real saved checkpoint path.


In [ ]:
if args.resume:
    print(crew.kickoff(from_checkpoint=CheckpointConfig(restore_from=str(args.resume))).raw)
else:
    print(crew.kickoff().raw)



## Bonus scenario — writing a book chapter by chapter
The idea behind resuming, without a model (inspired by CrewAI's *Write a Book* flow):
finished work is saved, so a re-run skips it. Run this cell twice.



In [ ]:
from workshop.common import ensure_runtime

book = ensure_runtime() / "book"
book.mkdir(exist_ok=True)
for number, title in enumerate(["Why agents", "Tools", "Flows"], start=1):
    chapter = book / f"chapter-{number}.txt"
    if chapter.exists():
        print("skip (already written):", title)
        continue
    chapter.write_text(f"Draft of {title}", encoding="utf-8")  # stands in for a slow Crew run
    print("wrote:", title)


## Failure mode to know about

Restoring a checkpoint can repeat a side effect performed just before interruption. Keep the action boundary idempotent independently of CrewAI checkpoints.


## Production implication

Test interruption after each task and directly before/after every side effect. Restore from real checkpoints and make external actions replay-safe.


## Your turn

Interrupt after task one, locate the saved checkpoint, and resume with its path.

**Checkpoint:** Completed research is reused and no mock action is duplicated.
